# CookBook

A notebook with an overview of all the main features and functions of the md library.

Use examples and techniques are categorized. You don't have to run all the cells sequentially — you only need to run the starting cell for the others to work correctly.


# Start

**run all cells of the section for proper operation**


**Imports**

Required imports. The library is two-tiered all functions are located in modules; they cannot be imported from the root (`md`).

The modules correspond to the following tasks:

- `chem` - managing base classes (trajectories, system, force field, water type, etc.)
- `simulations` - simulation, solvation, and other functions
- `mdp` - module with fields for MDP settings
- `analyse` - trajectory analysis module and everything else
- `running` - module for parallel and remote runs

In addition to the main modules, there is `pipelines` - a subpackage with predefined modules:

- `default_md` - predefined settings for standard Moldynamics runs
- `selections` - selection functions for various needs
- `fep` - FEP pipelines


In [ ]:
from pathlib import Path

from simdel import chem, func, run, analyse, traj, pipelines, sim

**Logging**

Use any custom log formatter for interception. All messages are sent from the `md` logger to a common channel with two additional parameters:

- `log_id` - the log entry number for the start and end of the command execution
- `log_group` - used in FEP to filter logs from different replicas


In [ ]:
WORKDIR = Path("temp_cookbook").resolve()
RAW_PROTEIN = Path("examples/data/CookBook/pfkfb3.pdb").resolve()
RAW_LIGAND = Path("examples/data/CookBook/lig.sdf").resolve()
FF_CUSTOM = Path("examples/data/CookBook/my.ff")
WORKDIR.mkdir(parents=True, exist_ok=True)

We can use the FEP log formatter and interceptor. Use inner log functions, only for example:


In [ ]:
from simdel.run import pipelines as pipelines_

pipelines_.handle_stdout_log()
pipelines_.handle_log_file(folder=Path(WORKDIR))

In [ ]:
FF = chem.DefaultFF.amber99sb_ildn
WATER_TYPE = chem.WaterType.tip3p

PROTEIN = func.parametrize_protein(
    geometry=RAW_PROTEIN,
    ff=FF,
    water_type=None,
    workdir=WORKDIR / "0_PROTEIN",
)
LIGAND = func.parametrize_small(
    sdf=RAW_LIGAND,
    ff=chem.DefaultFF.openff210,
    workdir=WORKDIR / "0_LIGAND",
    fast=True,
)

COMPLEX = PROTEIN + LIGAND
BOX = func.create_box(
    system=COMPLEX,
    workdir=WORKDIR / "0_BOX",
)
SOLVATED = func.solvate(
    system=BOX,
    ff=FF,
    water_type=WATER_TYPE,
    flexible_water=False,
    workdir=WORKDIR / "0_SOLVATED",
)
MDP = pipelines.em_mdp(emtol=1000)
IONIC = func.add_ions(
    system=SOLVATED,
    parameters=pipelines.add_ions_mdp(),
    concentration=0.15,
    positive_ion=chem.DefaultIon.Na,
    negative_ion=chem.DefaultIon.Cl,
    workdir=WORKDIR / "0_IONIC",
)
IONIC_INDEXED = IONIC.set_indexes(
    **func.create_gromacs_indexes(workdir=WORKDIR / "0_INDEX", system=IONIC)
)
sim1, *_ = func.simulate(
    workdir=WORKDIR / "0_SIMULATE-1",
    system=IONIC_INDEXED,
    parameters=pipelines.em_mdp(emtol=1000),
)
sim2, *_ = func.simulate(
    workdir=WORKDIR / "0_SIMULATE-2",
    system=sim1,
    parameters=pipelines.nvt_mdp(nsteps=1000),
)
SIM, SIM_TRAJ, SIM_ENERGY, _ = func.simulate(
    workdir=WORKDIR / "0_SIMULATE",
    system=sim2,
    parameters=pipelines.npt_mdp(nsteps=1000),
)  # type: ignore
SIM_TRAJ: chem.Trajectory

# Run module

**Using multithreaded contexts**


In addition to standard execution, functions can be wrapped in "pools" for parallelization and/or cluster execution.

They act as context managers; the function signature remains unchanged when wrapped, but they return a Future wrapper around the result, not the function's result.

Types:

- `LocalPool` - runs the function in separate local threads
- `RemotePool` - runs the function on the cluster in separate tasks

Until the `.result` method is called on the Future, the process will not wait for the result to complete.


### LocalPool


LocalPool makes it easy to run parallel tasks. It uses concurrent.futures.ThreadPoolExecutor internally.


In [ ]:
with run.LocalPool(
    func.parametrize_protein,
    max_workers=2,
) as f:
    futures: list[run.LocalFuture] = []
    for i in [chem.DefaultFF.amber99sb_ildn, chem.DefaultFF.amber99sb, chem.DefaultFF.amberGS]:
        futures.append(
            f(
                geometry=RAW_PROTEIN,
                workdir=WORKDIR / "localPool_parametrize" / i.name,
                ff=i,
            )
        )
results = [i.result() for i in futures]

### RemotePool

coming soon


# Chem module

**Base classes**


The main part of the md.chem module is the System.

chem.System is an immutable object; everything that can be done with it is represented in its methods. It works similarly to pandas.DataFrame: any method returns a modified copy of the object or a property.

In addition to fields with topology and geometry, the system contains markers that are used only to denote system properties.


In [ ]:
system = chem.System.load(
    gro=Path("cookbook/0_SIMULATE/sim.gro"),
    top=Path("cookbook/0_SIMULATE/sim.top"),
    index=Path("cookbook/0_SIMULATE/sim.ndx"),
)

To specify metadata - the type of force field and the type of water, the method is used


In [ ]:
system = system.set_info(
    ff_type=chem.DefaultFF.amber99sb_ildn,
    water_type=chem.WaterType.tip3p,
    water_flexibility=False,
)

### System properties


The system stores information about the environment—the force field, the solvent type—so it can validate the mixing of different systems and prevent the mixing of different force fields, water types, and so on.

> _However, since GROMACS force fields contain information not only about protein atoms but also about ionic atoms, and OpenFF fields contain ligand atom types, mixing GROMACS fields with OpenFF fields is possible._

The system representation contains information about the topologies it contains.

`<System: NAME [TOP1=X] (TOP2) ff= FIELD_TYPE WATER_TYPE-WATER_HARDNESS>`

- NAME - system name
- TOP1 - topology represented in geometry, X molecules
- TOP2 - topology is in the system's topology dictionary, but not in the geometry
- FIELD_TYPE - field name and version (if specified)
- WATER_TYPE - water type (tip3p...) if the system is solvated, otherwise `None`
- WATER_HARDNESS - whether the FLEXIBLE flag is applied to water (there can only be one water in the system) (if there is water)


In [ ]:
LIGAND

In [ ]:
PROTEIN

In [ ]:
# The names of the topologies of molecules are unique,
# the user has access to them via `system.topology_map`,
# but they are not used directly anywhere.
PROTEIN.topology_map, IONIC_INDEXED.topology_map

In [ ]:
# List of molecules in the order of their appearance in geometry
PROTEIN.molecules, IONIC_INDEXED.molecules[:3]

In [ ]:
# Geometry table
PROTEIN.geometry, IONIC_INDEXED.geometry

In [ ]:
# System index
PROTEIN.index, IONIC_INDEXED.index.keys()

In [ ]:
protein2 = PROTEIN.rename("new_protein")
print(PROTEIN.name, protein2.name)

In [ ]:
# Water type and  forcefield type information
PROTEIN.info

### Mixing


When adding systems, force fields are combined, topologies and geometries are added together.

- Error if the fields are incompatible: 2 GROMACS fields or openff fields with overlapping but not equal atom types

- Mixing if compatible - the more general field includes the less general one (openff + amber99sb-ildn = amber99sb-ildn)

- Geometry is automatically sorted (molecules with the same topology are grouped into 1 group, and the order of these groups is determined by the order of their entry into the dictionary of topologies)

**CAUTION:** One GROMACS system can be mixed with molecules parameterized by different openff fields, which can lead to errors and cannot be tracked.

---

Sort

```python
a.topology_map = {'SOL': sol_top,'A':a_top}
a.molecules = ['SOL', 'A']

b.topology_map = {'A':a_top 'B': b_top}
b.molecules = ['B', 'A']

c=a+b

=> c.topology_map= {'SOL': sol_top,'A':a_top,'B': b_top}
=> c.molecules = ['SOL', 'A','A','B']
```


In [ ]:
system = PROTEIN + LIGAND

In [ ]:
system

In [ ]:
system.topology_map

In [ ]:
system.molecules

### Geometry view


To analyze atoms in geometry and construct bit masks based on their properties, you can use a geometry mapping (a summary table of geometry atoms), which is generated anew each time.

It is a typed pandas.DataFrame table


In [ ]:
v = SOLVATED.geometry_view
v

In [ ]:
v[v.molecule == "SOL"]

### Topology view


To analyze atoms in topologies (there are no positions, velocities, each topology occurs 1 time) and construct bit masks based on their properties (so far only for each atom, parameters for pairs, etc. are not summed up), you can use topology mapping (a summary table for atoms in topologies), it is generated anew each time.

It is a typed pandas.DataFrame table


In [ ]:
v = SOLVATED.topology_view
v

In [ ]:
v[v.molecule == "SOL"]

### Indexes


Indexes are created in the system using binary masks for displaying **geometry**. You can perform all operations on them AND OR NOT, etc. to build the desired mask.

If the geometry of the system changes, the indexes are reset.:

- Addition of systems
- Addition of ions, solvation

They remain:

- after the usual simulation, `func.simulate' (only coordinates and speeds change)
- rename, set_water_type (system parameters change)
- extend_topologies, rename_topology, remove_topologies (topologies change, but not geometries)


You can use the `md.pipelines` module as a collection of standard masks: highlight molecules, highlight atoms .


In [ ]:
v = SOLVATED.geometry_view
mask = pipelines.select_solvent(v, include_hydrogens=False)
mask2 = pipelines.select_solvent(v) * ~pipelines.select_H(v)
print(all(mask2 == mask))
v[mask]

In [ ]:
SOLVATED.set_indexes(solvent=pipelines.select_solvent(v)).save(WORKDIR / "index")

In [ ]:
solvated2 = chem.System.load(
    gro=WORKDIR / "index" / f"{SOLVATED.name}.gro",
    top=WORKDIR / "index" / f"{SOLVATED.name}.top",
    index=WORKDIR / "index" / f"{SOLVATED.name}.ndx",
)

In [ ]:
solvated2.index.keys()

The index can be saved separately.


In [ ]:
func.dump_index(
    index_file=WORKDIR / "index" / "sol.ndx",
    indexes=dict(SOL=pipelines.select_solvent(v)),
)

### Restraints


The previously used POSRES and WATER_POSRES settings are not working now - you need to specify which atoms in the topology should have which constraint parameters.

Constraints are set using special classes that are created from mapping **topologies**. If there are limitations in the system, the data will be presented in the table

To set a restriction, you need to set the parameters in a special class of this restriction. By default, the class reads constraints from the system, the fields `molecule`, `ai` are filled in for all atoms, the other fields in the row should be:

1. All other fields in the row are empty - there are no restrictions (or restrictions reset)
2. All other fields are filled in - there is a restriction

Variants:

- **Positional Restraints**


You can use the `md.selections` module like a collection of standard masks: highlight molecules, highlight atoms. There are also special functions for easily defining constraints.

> Some preset selection functions cannot be applied to the topology display, but only to the geometry display - see the function description.


In [ ]:
v = SOLVATED.topology_view
posres = chem.PositionRestraints.from_top_view(v)
posres

In [ ]:
# Не забудьте параметр ftype
mask = (v.molecule == "Protein_chain_A") & (v.atomic_number == 6)
cols = ["ftype", "x", "y", "z"]
posres[mask, cols] = (1, 900, 900, 900)
posres

In [ ]:
mask = pipelines.select_C(v) & pipelines.select_molecules(
    view=v, molecule_names=["Protein_chain_A"]
)
posres2 = pipelines.set_constant_posres(view=v, selection=mask, value=900)
posres2 == posres

In [ ]:
ionic_posres1 = func.set_restraints(system=SOLVATED, restraints=posres)
ionic_posres1.save(WORKDIR / "restraints_posres")

Installing empty constraints in the system removes all constraints.


In [ ]:
clear_posres = chem.PositionRestraints.from_top_view(v)
clear_posres

In [ ]:
clear_posres.clear()
ionic_posres_clear = func.set_restraints(system=ionic_posres1, restraints=clear_posres)
ionic_posres_clear.save(WORKDIR / "restraints_clearPosres")

### Forcefield


In [ ]:
# Standard force fields, folders of which are copied to the library are presented in the classroom
chem.DefaultFF.__annotations__

In [ ]:
# Types of water that are present in force fields
chem.WaterType

In [ ]:
# Uploading your GROMACS force field
gromacs_ff = chem.GromacsFF.load(ff=Path("examples/data/MD/my.ff"))

# Uploading your Openff force field
openff_ff = chem.OpenFF.load(name="openff_no_water-3.0.0-alpha0.offxml")

In [ ]:
# Inside the GROMACS field there is a map of water types and their geometries
gromacs_ff.water_map
# You can check the compatibility of water and force field
gromacs_ff.get_water_info(chem.WaterType.spc)

# Func module

**Simul**


### Protein parameterization


You can parameterize the system with water and ions, or not specify the type of water at all if it is not present in the system with protein. Water can be added later. If there is no water in the system, there will be no topology.

You can use your own field

> If there are no water molecules in the system, there will be no water topologies in the system anyway


In [ ]:
ff = chem.GromacsFF.load(FF_CUSTOM)
protein = func.parametrize_protein(
    geometry=RAW_PROTEIN,
    workdir=WORKDIR / "parametrize_protein",
    ff=ff,
    fix_missing=True,
    water_type=None,
)

### Parameterization of the ligand


You can use any force field that is supported by Openff.


In [ ]:
ff = chem.OpenFF.load("openff-2.1.0")
ligand = func.parametrize_small(
    sdf=RAW_LIGAND,
    workdir=WORKDIR / "0_ligand",
    fast=True,
    ff=ff,
)

### Assembling the box


Setting the type and borders of the box


In [ ]:
box = func.create_box(
    system=COMPLEX,
    workdir=WORKDIR / "box",
)
box.geometry

### Solvation


Creation of the solvate shell of the system (water + ions):

- adds the topology of water to the list of topologies and water molecules to the geometry by setting water type flags
- adds only ion topologies

**CAUTION:** The type of water must correspond to the force field (see `chem.WATER_TYPE_MAP`), does not apply to custom fields

> The system can only be solvated if there is no water in the system


In [ ]:
solvated = func.solvate(
    system=BOX,
    ff=FF,
    water_type=chem.WaterType.tip3p,
    flexible_water=True,
    workdir=WORKDIR / "solvate",
)

### Replacement of the water shell


To speed up calculations, it may be necessary to change the flexible water to a rigid water model. Or a replacement for another type of water.

Limitations:

- You can replace water types with only one geometry (see `chem.WATER_GEOMETRY_MAP`)
- The type of water must match the force field

> The water shell can be replaced if it has already been set (water must be in the system)


In [ ]:
SOLVATED.info.water_type

In [ ]:
resolvated = func.resolvate(
    workdir=WORKDIR / "resolvate",
    system=SOLVATED,
    water_type=chem.WaterType.spc,
    flexible_water=False,
)

### Adding ions


Adding ions. You can add only those ions that are present in the topology of the system. The basic ion types are represented in the `DefaultIon` class


In [ ]:
na = chem.DefaultIon.Na
cl = chem.Ion(name="CL", charge=-1)

In [ ]:
ionic = func.add_ions(
    system=SOLVATED,
    positive_ion=na,
    negative_ion=cl,
    parameters=pipelines.add_ions_mdp(),
    concentration=0.15,
    workdir=WORKDIR / "ionic",
)

### Simulation


You can run any simulation.


In [ ]:
em_system, em_traj, energy, plumed_out = func.simulate(
    workdir=WORKDIR / "simulate",
    system=ionic_posres_clear,
    parameters=MDP,
)

# Sim module

**A class with all mdp parameters**


The configuration for the simulation can be set arbitrarily using `md.sim.MDP`. All fields are typed

The `md.pipelines.default_md` module contains standard settings for simulations and auxiliary functions for molecular dynamics.


In [ ]:
emtol = 1000
emstep = 0.001

In [ ]:
my_mdp = sim.GromacsSimulator(
    name="em",
    # Run control
    integrator=sim.Integrator.Steep,
    # EM
    emtol=emtol,
    emstep=emstep,
    # TI
    nsteps=-1,
    # Neighbor searching
    nstlist=1,
    cutoff_scheme=sim.CutoffScheme.Verlet,
    pbc=sim.PBC.XYZ,
    # Electrostatics
    coulombtype=sim.CoulombType.PME,
    rcoulomb=1.2,
    # Van der Waals
    dispcorr=sim.DispCorr.EnerPres,
    rvdw=1.2,
    # Ewald summing
    pme_order=4,
    fourierspacing=0.16,
)
pattern_mdp = pipelines.em_mdp(emtol=emtol, emstep=emstep)

In [ ]:
dict(pattern_mdp) == dict(my_mdp)

# Traj module

**Functions of working with trajectories**


Allfunctions for working with trajectories are presented in the md.traj module.

A trajectory is a set of snapshots of the geometry of a system that cannot exist without the system, but is not syntactically related to it in any way - you can change the system as you like (if the geometry does not change): rename topologies, delete unnecessary ones, and so on. In this case, the trajectory will still correspond to the system.


In [ ]:
trajectory = chem.Trajectory(
    file=Path("cookbook/0_SIMULATE/sim.xtc"),
    frames=3,
    dt=1,
)

Get mdtraj trajectory


In [ ]:
system = chem.System.load(
    gro=Path("cookbook/0_SIMULATE/sim.gro"),
    top=Path("cookbook/0_SIMULATE/sim.top"),
    index=Path("cookbook/0_SIMULATE/sim.ndx"),
)
# для использования mdtraj.Trajectory нужна система
md_traj = trajectory.get_mdtraj(system=system)

### Functions


Dividing the trajectory into separate frames


In [ ]:
trajectory_splitted = traj.split(
    trajectory=SIM_TRAJ,
    system=SIM,
    workdir=WORKDIR / "split_traj",
    segments=2,
    start_time=0,
    end_time=3,
    name="",
    compress=False,
)

Extract 1 frame from the trajectory


In [ ]:
frame = traj.extract_frame(
    trajectory=SIM_TRAJ,
    system=SIM,
    workdir=WORKDIR / "extract_frame",
    time=1,
    name="",
    compress=False,
)

Correction of the trajectory box centering (while all settings are in 1 function)


In [ ]:
v = SIM.geometry_view
mask = v.molecule == "Protein"

trajectory_splitted = traj.fix(
    trajectory=SIM_TRAJ,
    system=SIM,
    workdir=WORKDIR / "fix_traj",
    center_location=traj.CenterLocation.Zero,
    center_target=mask,
    pbc_algorithm=traj.PBCAlgorithm.Molecule,
    box_type=traj.BoxType.Rectangle,
    box_size=(2, 2, 2),
    name="",
    compress=False,
)

Shift the trajectory to a vector


In [ ]:
trajectory_shifted = traj.shift_on_vector(
    trajectory=SIM_TRAJ,
    system=SIM,
    workdir=WORKDIR / "shift_traj",
    center_location=traj.CenterLocation.Zero,
    center_target=mask,
    name="",
    compress=False,
)

Adjust the trajectory to the desired structure


In [ ]:
trajectory_shifted = traj.fit(
    trajectory=SIM_TRAJ,
    system=SIM,
    workdir=WORKDIR / "fit_traj",
    fit_type=traj.FitType.Trans,
    fit_target=mask,
    center_location=traj.CenterLocation.Zero,
    center_target=mask,
    name="",
    compress=False,
)